In [1]:
import pandas as pd 
import numpy as np 


# Customer _d & Cities 

In [3]:
np.random.seed(42)
n_customers = 50000 

customer_id = [f"CUST{100000+i}" for i in range(n_customers)]
cities = np.random.choice(
    ['Delhi', 'Gurgaon', 'Mumbai', 'Bangalore', 'Noida', 'Pune'],
    size=n_customers,
    p=[0.20, 0.20, 0.20, 0.15, 0.15, 0.10]
)

# Ages & Gender

In [4]:
ages = np.random.randint(18,66, size=n_customers)

gender = np.random.choice(
    ['Male','Female','Other'],
    size= n_customers, 
    p = [0.52,0.46,0.02]
)

# Plan Type 

In [5]:
def assign_plan(age):
    if age < 30:
        return np.random.choice(['Prepaid', 'Postpaid', 'Broadband'], p=[0.70, 0.20, 0.10])
    elif age < 45:
        return np.random.choice(['Prepaid', 'Postpaid', 'Broadband'], p=[0.40, 0.35, 0.25])
    else:
        return np.random.choice(['Prepaid', 'Postpaid', 'Broadband'], p=[0.25, 0.40, 0.35])

plan_type = np.array([assign_plan(age) for age in ages])

# Contract Type 

In [6]:
def assign_contract(plan):
    if plan == 'Prepaid':
        return 'No Contract'
    else:
        return np.random.choice(['Month-to-Month', '1-Year', '2-Year'], p=[0.50, 0.30, 0.20])

contract_type = np.array([assign_contract(plan) for plan in plan_type])

# Tenure Months + Monthly Charges

In [7]:
def assign_tenure(contract):
    if contract == 'No Contract':
        return np.random.randint(1, 37)
    elif contract == 'Month-to-Month':
        return np.random.randint(1, 25)
    elif contract == '1-Year':
        return np.random.randint(6, 49)
    else:
        return np.random.randint(12, 73)

tenure_months = np.array([assign_tenure(c) for c in contract_type])

def assign_charges(plan):
    if plan == 'Prepaid':
        return round(np.random.uniform(200, 500), 2)
    elif plan == 'Postpaid':
        return round(np.random.uniform(500, 1500), 2)
    else:
        return round(np.random.uniform(800, 2500), 2)

monthly_charges = np.array([assign_charges(p) for p in plan_type])

# Create Dataframe

In [8]:
customers = pd.DataFrame({
    'customer_id': customer_id,
    'city': cities,
    'age': ages,
    'gender': gender,
    'plan_type': plan_type,
    'contract_type': contract_type,
    'tenure_months': tenure_months,
    'monthly_charges': monthly_charges
})

customers.head()

,customer_id,city,age,gender,plan_type,contract_type,tenure_months,monthly_charges
0,CUST100000,Gurgaon,26,Male,Postpaid,2-Year,28,1388.77
1,CUST100001,Pune,53,Male,Postpaid,Month-to-Month,4,876.38
2,CUST100002,Bangalore,23,Female,Prepaid,No Contract,4,289.27
3,CUST100003,Mumbai,41,Male,Prepaid,No Contract,5,365.02
4,CUST100004,Delhi,43,Female,Postpaid,Month-to-Month,16,1160.59


In [9]:
print(customers['city'].value_counts(normalize=True))
print()
print(customers['plan_type'].value_counts(normalize=True))
print()
print(customers['contract_type'].value_counts(normalize=True))

city
Gurgaon      0.20160
Delhi        0.20006
Mumbai       0.19932
Bangalore    0.15084
Noida        0.14864
Pune         0.09954
Name: proportion, dtype: float64

plan_type
Prepaid      0.40294
Postpaid     0.33928
Broadband    0.25778
Name: proportion, dtype: float64

contract_type
No Contract       0.40294
Month-to-Month    0.29792
1-Year            0.17960
2-Year            0.11954
Name: proportion, dtype: float64


In [10]:
customers.to_csv(r"C:\Data Analytics\Projects\END TO END\TeleCom_Analysis\Dataset\Customers.csv", index=False)

# Churn Status Generation

In [11]:
def churn_probability(contract, tenure):
    if contract == 'No Contract':
        base_prob = 0.22
    elif contract == 'Month-to-Month':
        base_prob = 0.28
    elif contract == '1-Year':
        base_prob = 0.12
    else:
        base_prob = 0.05

    if tenure < 6:
        base_prob += 0.10
    elif tenure < 12:
        base_prob += 0.05

    return min(base_prob, 0.60)

churn_prob = np.array([churn_probability(c, t) for c, t in zip(contract_type, tenure_months)])
is_churned = np.random.binomial(1, churn_prob)

In [12]:
print("Overall Churn Rate:", is_churned.mean())
print()

check_df = pd.DataFrame({
    'contract_type': contract_type,
    'is_churned': is_churned
})
print(check_df.groupby('contract_type')['is_churned'].mean())

Overall Churn Rate: 0.2185

contract_type
1-Year            0.125612
2-Year            0.048185
Month-to-Month    0.311090
No Contract       0.241972
Name: is_churned, dtype: float64


# Churn Date & Churn Reason

In [13]:
churn_reasons_list = ['High Charges', 'Poor Network', 'Better Offer Elsewhere', 'Poor Customer Service', 'Other']

churn_date = []
churn_reason = []

for churned in is_churned:
    if churned == 1:
        random_days_ago = np.random.randint(1, 181)
        date = pd.Timestamp('2026-01-31') - pd.Timedelta(days=random_days_ago)
        churn_date.append(date)
        churn_reason.append(np.random.choice(churn_reasons_list, p=[0.30, 0.25, 0.20, 0.15, 0.10]))
    else:
        churn_date.append(pd.NaT)
        churn_reason.append(np.nan)

churn_date = np.array(churn_date)
churn_reason = np.array(churn_reason)

In [14]:
check_df2 = pd.DataFrame({
    'is_churned': is_churned,
    'churn_date': churn_date,
    'churn_reason': churn_reason
})

print(check_df2[check_df2['is_churned'] == 1].head())
print()
print(check_df2[check_df2['is_churned'] == 0].head())

    is_churned churn_date            churn_reason
2            1 2025-10-03  Better Offer Elsewhere
5            1 2025-12-11  Better Offer Elsewhere
9            1 2025-11-15                   Other
17           1 2025-11-22            High Charges
20           1 2025-10-10            Poor Network

   is_churned churn_date churn_reason
0           0        NaT          nan
1           0        NaT          nan
3           0        NaT          nan
4           0        NaT          nan
6           0        NaT          nan


In [15]:
churn_status = pd.DataFrame({
    'customer_id': customer_id,
    'is_churned': is_churned,
    'churn_date': churn_date,
    'churn_reason': churn_reason
})

churn_status['churn_reason'] = churn_status['churn_reason'].replace('nan', np.nan)

churn_status.head()

,customer_id,is_churned,churn_date,churn_reason
0,CUST100000,0,NaT,NaN
1,CUST100001,0,NaT,NaN
2,CUST100002,1,2025-10-03,Better Offer Elsewhere
3,CUST100003,0,NaT,NaN
4,CUST100004,0,NaT,NaN


In [16]:
churn_status.to_csv(r"C:\Data Analytics\Projects\END TO END\TeleCom_Analysis\Dataset\churn_status.csv", index=False)

# Usage Data — Base Setup

In [17]:
months_list = pd.date_range(start='2025-08-01', end='2026-01-01', freq='MS')
print(months_list)

DatetimeIndex(['2025-08-01', '2025-09-01', '2025-10-01', '2025-11-01',
               '2025-12-01', '2026-01-01'],
              dtype='datetime64[ns]', freq='MS')


# Base Usage Averages Per Customer

In [18]:
def base_data_usage(plan):
    if plan == 'Prepaid':
        return np.random.uniform(2, 15)
    elif plan == 'Postpaid':
        return np.random.uniform(5, 30)
    else:
        return np.random.uniform(20, 100)

def base_calls(plan):
    if plan == 'Prepaid':
        return np.random.uniform(100, 500)
    elif plan == 'Postpaid':
        return np.random.uniform(300, 1000)
    else:
        return np.random.uniform(0, 50)

avg_data_usage = np.array([base_data_usage(p) for p in plan_type])
avg_calls = np.array([base_calls(p) for p in plan_type])

In [19]:
records = []

for i in range(n_customers):
    cust_id = customer_id[i]
    base_data = avg_data_usage[i]
    base_call = avg_calls[i]
    churned = is_churned[i]

    for month_num, month_date in enumerate(months_list):
        if churned == 1 and month_num >= 3:
            decline_factor = 1 - ((month_num - 2) * 0.20)
        else:
            decline_factor = 1.0

        decline_factor = max(decline_factor, 0.1)

        data_gb = round(base_data * decline_factor * np.random.uniform(0.85, 1.15), 2)
        calls_min = round(base_call * decline_factor * np.random.uniform(0.85, 1.15), 2)
        sms = np.random.randint(0, 100)
        recharge = round(monthly_charges[i] * np.random.uniform(0.9, 1.1), 2)

        records.append([cust_id, month_date, data_gb, calls_min, sms, recharge])

usage_data = pd.DataFrame(records, columns=['customer_id', 'month', 'data_used_gb', 'calls_minutes', 'sms_count', 'recharge_amount'])

In [20]:
print("Shape:", usage_data.shape)
print()

sample_churned_id = churn_status[churn_status['is_churned']==1]['customer_id'].iloc[0]
print(usage_data[usage_data['customer_id']==sample_churned_id])


Shape: (300000, 6)

   customer_id      month  data_used_gb  calls_minutes  sms_count  \
12  CUST100002 2025-08-01          7.03         324.12         37   
13  CUST100002 2025-09-01          7.87         316.94         16   
14  CUST100002 2025-10-01          6.93         328.39         62   
15  CUST100002 2025-11-01          6.17         240.90         58   
16  CUST100002 2025-12-01          5.30         173.13         42   
17  CUST100002 2026-01-01          3.08         108.77         43   

    recharge_amount  
12           286.71  
13           292.21  
14           304.94  
15           298.39  
16           307.62  
17           306.69  


In [21]:
usage_data.to_csv(r"C:\Data Analytics\Projects\END TO END\TeleCom_Analysis\Dataset\usage_data.csv", index=False)

# Complaints — Number of Complaints Per Customer

In [22]:
def num_complaints(churned):
    if churned == 1:
        return np.random.choice([0, 1, 2, 3, 4, 5], p=[0.10, 0.15, 0.20, 0.25, 0.20, 0.10])
    else:
        return np.random.choice([0, 1, 2, 3, 4, 5], p=[0.40, 0.30, 0.15, 0.10, 0.04, 0.01])

complaint_counts = np.array([num_complaints(c) for c in is_churned])

print("Avg complaints - Churned:", complaint_counts[is_churned==1].mean())
print("Avg complaints - Non-Churned:", complaint_counts[is_churned==0].mean())

Avg complaints - Churned: 2.596887871853547
Avg complaints - Non-Churned: 1.1058989123480487


In [23]:
complaint_types_list = ['Billing Issue', 'Network Issue', 'Service Quality', 'Data/Speed Issue', 'Other']

complaint_records = []
complaint_id_counter = 1

for i in range(n_customers):
    cust_id = customer_id[i]
    n_complaints = complaint_counts[i]
    churned = is_churned[i]

    for j in range(n_complaints):
        c_id = f"COMP{100000 + complaint_id_counter}"
        complaint_id_counter += 1

        days_ago = np.random.randint(1, 181)
        c_date = pd.Timestamp('2026-01-31') - pd.Timedelta(days=days_ago)

        c_type = np.random.choice(complaint_types_list, p=[0.25, 0.25, 0.20, 0.20, 0.10])

        if churned == 1:
            status = np.random.choice(['Resolved', 'Pending'], p=[0.60, 0.40])
            res_days = np.random.randint(3, 15)
        else:
            status = np.random.choice(['Resolved', 'Pending'], p=[0.85, 0.15])
            res_days = np.random.randint(1, 7)

        complaint_records.append([c_id, cust_id, c_date, c_type, status, res_days])

complaints = pd.DataFrame(complaint_records, columns=['complaint_id', 'customer_id', 'complaint_date', 'complaint_type', 'resolution_status', 'resolution_days'])

In [24]:
complaints.shape

(71584, 6)

In [25]:
merged_check = complaints.merge(churn_status[['customer_id', 'is_churned']], on='customer_id')

print(merged_check.groupby('is_churned')['resolution_days'].mean())
print()
print(merged_check.groupby('is_churned')['resolution_status'].value_counts(normalize=True))

is_churned
0    3.495152
1    8.497127
Name: resolution_days, dtype: float64

is_churned  resolution_status
0           Resolved             0.851063
            Pending              0.148937
1           Resolved             0.597476
            Pending              0.402524
Name: proportion, dtype: float64


In [27]:
complaints.to_csv(r"C:\Data Analytics\Projects\END TO END\TeleCom_Analysis\Dataset\complaints.csv", index=False)